In [0]:
%pip install xgboost

In [0]:
import pandas as pd
import numpy as np
import pickle
import boto3
import json
from io import BytesIO

# ML Imports
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor 

# 1. Cargar Credenciales
with open("aws_secrets.json", "r") as f:
    config = json.load(f)

# 2. Conexión a Spark (ESTRATEGIA GOLD PARQUET)
# Usamos la carpeta 'app_inmuebles' que es Parquet puro y no falla por _delta_log
print("⏳ Cargando datos Gold (Parquet) para entrenamiento...")
ruta_gold = f"s3a://{config['bucket_name']}/gold/app_inmuebles/"

# CAMBIO CLAVE: .format("parquet") en lugar de "delta"
df_spark = spark.read.format("parquet") \
    .option("fs.s3a.access.key", config['aws_access_key']) \
    .option("fs.s3a.secret.key", config['aws_secret_key']) \
    .option("fs.s3a.endpoint", "s3.amazonaws.com") \
    .load(ruta_gold)

# Validamos columnas disponibles (Gold ya tiene nombres limpios)
print(f"📊 Columnas encontradas: {df_spark.columns}")

# Seleccionamos y pasamos a Pandas
# Gold ya tiene 'barrio', 'titulo', 'precio_num', 'area_m2'
df = df_spark.select("precio_num", "area_m2", "barrio", "titulo").dropna().toPandas()

# 3. Ingeniería de Características
df.rename(columns={'barrio': 'ubicacion'}, inplace=True)
df['texto_completo'] = df['ubicacion'] + " " + df['titulo']

# --- FILTRADO DE OUTLIERS ---
df_clean = df[
    (df["precio_num"] > df["precio_num"].quantile(0.05)) & 
    (df["precio_num"] < df["precio_num"].quantile(0.95)) & 
    (df['area_m2'] > 15) & 
    (df['area_m2'] < 1000)
]

print(f"🧹 Set de Entrenamiento: {len(df_clean)} registros.")

X = df_clean[['area_m2', 'texto_completo']]
y = df_clean['precio_num']

# 4. Pipeline XGBoost (Igual que antes, porque es excelente)
preprocessor = ColumnTransformer(
    transformers=[
        ('txt', TfidfVectorizer(max_features=300, stop_words=['en', 'de', 'la', 'el', 'y', 'con', 'se', 'del']), 'texto_completo'),
        ('num', 'passthrough', ['area_m2'])
    ]
)

xgboost_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', TransformedTargetRegressor(regressor=xgboost_model, func=np.log1p, inverse_func=np.expm1))
])

# 5. Entrenamiento
print("🚀 Entrenando XGBoost...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model_pipeline.fit(X_train, y_train)

# Métricas
score_r2 = model_pipeline.score(X_test, y_test)
from sklearn.metrics import mean_absolute_error
y_pred = model_pipeline.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)

print(f"\n📊 RESULTADOS:")
print(f"   🌟 R2 Score: {score_r2:.4f}")
print(f"   💰 MAE: ${mae:,.0f} COP")

# 6. Guardar Modelo en S3
if score_r2 > 0.0: # Guardamos aunque sea bajo para probar la app
    print("✅ Guardando modelo en S3...")
    bucket_name = config['bucket_name']
    file_key = "models/modelo_xgboost_v1.pkl"
    
    buffer = BytesIO()
    pickle.dump(model_pipeline, buffer)
    buffer.seek(0)
    
    s3 = boto3.client('s3', 
        aws_access_key_id=config['aws_access_key'],
        aws_secret_access_key=config['aws_secret_key']
    )
    s3.upload_fileobj(buffer, bucket_name, file_key)
    print(f"💾 Guardado exitoso: s3://{bucket_name}/{file_key}")